<a href="https://colab.research.google.com/github/dwmhr12/Nike_Run_Club_Tugas/blob/main/01_nike_run_club_reviews_scrapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Scraping Review Aplikasi Nike Run Club

## Aplikasi yang Di-Scraping
- **Nama Aplikasi:** Nike Run Club - Running Coach
- **Link Google Play:** [https://play.google.com/store/apps/details?id=com.nike.plusgps&hl=en](https://play.google.com/store/apps/details?id=com.nike.plusgps&hl=en)
- **ID Aplikasi:** com.nike.plusgps

## Tujuan
Mengambil data review dari aplikasi **Nike Run Club** di Google Play Store, memfilter review berbahasa Inggris, dan menyimpannya ke dalam format CSV untuk keperluan analisis lanjutan (Exploratory Data Analysis, Preprocessing, dan Sentiment Analysis).

## Deskripsi Proses
1. **Data Scraping:**
   - Menggunakan library `google_play_scraper` untuk mengambil seluruh review dari aplikasi Nike Run Club.
   - Review diambil dalam bahasa Inggris (`lang='en'`) dan diurutkan berdasarkan review terbaru (`Sort.NEWEST`).
   - Mengatur delay request untuk mencegah banned dari server.

2. **Filtering Bahasa:**
   - Walaupun sudah memilih bahasa Inggris saat scraping, tetap dilakukan deteksi ulang menggunakan library `langdetect` untuk memastikan review benar-benar berbahasa Inggris.
   - Menghapus review kosong atau yang tidak terdeteksi bahasanya.

3. **Output Dataset:**
   - **Jumlah total review berhasil dikumpulkan: 104,211 review**
   - Data hasil filtering disimpan ke file **`nike_reviews.csv`**.
   - Dataset siap digunakan untuk analisis lanjutan (preprocessing teks, sentiment analysis, klasifikasi, dll).

## Informasi Dataset
- Jumlah data: **104,211 baris / review**
- Kolom yang tersedia:
  - `reviewId`: ID unik review
  - `userName`: Nama pengguna
  - `userImage`: URL foto profil pengguna
  - `content`: Isi teks review
  - `score`: Rating bintang (1-5)
  - `thumbsUpCount`: Jumlah like pada review
  - `reviewCreatedVersion`: Versi aplikasi saat review dibuat
  - `at`: Tanggal review dibuat
  - `replyContent`: Isi balasan developer
  - `repliedAt`: Tanggal balasan developer
  - `appVersion`: Versi aplikasi

In [14]:
# 01-nike-run-club-reviews-scrapping.ipynb
# Tujuan: Mengambil review aplikasi Nike Run Club dari Google Play Store, memfilter review berbahasa Inggris, dan menyimpan ke CSV
# Catatan: Filtering bahasa Inggris dilakukan untuk memastikan analisis sentimen dan preprocessing teks lebih akurat, karena model NLP yang digunakan (TextBlob, TF-IDF) dioptimalkan untuk bahasa Inggris.

# Instalasi library yang diperlukan
try:
    import google_play_scraper
except ImportError:
    !pip install google_play_scraper
try:
    import langdetect
except ImportError:
    !pip install langdetect
try:
    import tqdm
except ImportError:
    !pip install tqdm

# Impor library yang diperlukan
import pandas as pd
import numpy as np
from google_play_scraper import reviews_all, Sort
from langdetect import detect, LangDetectException
from tqdm import tqdm
import os
import time

# Mengambil semua review dari aplikasi Nike Run Club
try:
    start_time = time.time()
    nike_reviews = reviews_all(
        'com.nike.plusgps',  # ID aplikasi Nike Run Club
        sleep_milliseconds=100,  # Jeda 100ms untuk keamanan
        lang='en',  # Bahasa Inggris (preferensi)
        sort=Sort.NEWEST  # Urutkan berdasarkan yang terbaru
    )
    print(f"Waktu scraping: {time.time() - start_time:.2f} detik")
except Exception as e:
    print(f"Error saat scraping: {e}")
    nike_reviews = []

# Konversi hasil scraping ke DataFrame
if not nike_reviews:
    print("Tidak ada review yang berhasil di-scraping.")
else:
    df_nike_reviews = pd.DataFrame(np.array(nike_reviews), columns=['content'])
    df_nike_reviews = df_nike_reviews.join(pd.DataFrame(df_nike_reviews.pop('content').tolist()))

    # Menampilkan jumlah total review yang di-scraping
    print(f"Jumlah total data yang di-scraping: {df_nike_reviews.shape[0]}")

    # Fungsi untuk mendeteksi bahasa
    def detect_language(text):
        try:
            return detect(text)
        except LangDetectException:
            common_english_words = {'good', 'great', 'bad', 'love', 'hate', 'app', 'run', 'awesome'}
            if any(word in text.lower() for word in common_english_words):
                return 'en'
            return 'unknown'

    # Pastikan kolom 'content' tidak memiliki NaN dan dikonversi ke string
    df_nike_reviews['content'] = df_nike_reviews['content'].astype(str).fillna("")

    # Hapus review kosong
    initial_count = df_nike_reviews.shape[0]
    df_nike_reviews = df_nike_reviews[df_nike_reviews['content'].str.strip() != ""]
    empty_count = initial_count - df_nike_reviews.shape[0]
    print(f"Jumlah review kosong yang dihapus: {empty_count}")

    # Tambahkan kolom bahasa dengan progress bar
    tqdm.pandas()
    df_nike_reviews['language'] = df_nike_reviews['content'].progress_apply(detect_language)

    # Hitung review non-Inggris
    non_english_count = df_nike_reviews[df_nike_reviews['language'] != 'en'].shape[0]
    print(f"Jumlah review non-Inggris yang dihapus: {non_english_count}")

    # Filter hanya review berbahasa Inggris
    df_nike_reviews = df_nike_reviews[df_nike_reviews['language'] == 'en']

    # Hapus kolom 'language' setelah filtering
    df_nike_reviews = df_nike_reviews.drop(columns=['language'])

    # Menampilkan jumlah review berbahasa Inggris
    print(f"Jumlah review berbahasa Inggris: {df_nike_reviews.shape[0]}")

    # Menampilkan 5 baris pertama dari DataFrame
    print("\nContoh 5 baris pertama dari data (bahasa Inggris):")
    display(df_nike_reviews.head())

    # Menampilkan sampel acak
    print("\nSampel acak 10 review berbahasa Inggris:")
    display(df_nike_reviews[['content', 'score']].sample(10))

    # Menampilkan statistik dasar
    print("\nStatistik Dasar Data:")
    print(f"Kolom yang di-scraping: {df_nike_reviews.columns.tolist()}")
    print(f"Rentang tanggal review: {df_nike_reviews['at'].min()} hingga {df_nike_reviews['at'].max()}")
    print("\nDistribusi Skor Review:")
    print(df_nike_reviews['score'].value_counts().sort_index())

    # Menyimpan DataFrame ke file CSV
    file_path = 'nike_reviews.csv'
    if os.path.exists(file_path):
        print(f"File {file_path} sudah ada. Tidak akan ditimpa.")
    else:
        df_nike_reviews.to_csv(file_path, index=False, encoding='utf-8')
        print(f"DataFrame telah disimpan ke {file_path}")

Waktu scraping: 71.97 detik
Jumlah total data yang di-scraping: 127038
Jumlah review kosong yang dihapus: 0


100%|██████████| 127038/127038 [10:16<00:00, 205.90it/s]


Jumlah review non-Inggris yang dihapus: 22827
Jumlah review berbahasa Inggris: 104211

Contoh 5 baris pertama dari data (bahasa Inggris):


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,79941129-e19a-4e9b-b396-935b35278463,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,"this app stops recording your run midway ,does...",1,0,4.66.0,2025-06-13 07:19:01,None,NaT,4.66.0
1,ce66e4ca-9852-4549-9713-7dbe40b8598a,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,best run tracking app out there ❤️‍🩹💯💯,5,0,4.66.0,2025-06-13 04:41:58,None,NaT,4.66.0
3,ba005be3-5bea-4a9a-bfef-909c7a7591c9,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,Heart rate monitor not working on Galaxy Watch...,2,0,2.9.0,2025-06-12 22:19:07,None,NaT,2.9.0
4,db3bbe8e-0718-4e30-a0c4-f26bf1af8b9f,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,Had this app years ago and logged in to all of...,1,0,4.66.0,2025-06-12 19:21:16,None,NaT,4.66.0
5,8e019d0f-54cc-4613-96b8-945072433e0f,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,Why do I need to login to my account every mor...,1,2,4.66.0,2025-06-12 13:15:16,None,NaT,4.66.0



Sampel acak 10 review berbahasa Inggris:


,content,score
3555,I have a half-marathon coming up in less than ...,1
56101,NRC+ helped me run my first 10 k marathon. Lov...,5
108771,It's very good,5
58756,"Great running at that charts mileage, weather,...",5
94454,Better than RunKeeper app,4
73306,How to save run on this app,1
14863,Been using this app for a while but it's not w...,1
28628,The app is not bad. It's just not good enough ...,3
103637,A very useful app for jogging. Wish it had a w...,4
30777,Good app,5



Statistik Dasar Data:
Kolom yang di-scraping: ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']
Rentang tanggal review: 2012-06-21 18:18:07 hingga 2025-06-13 07:19:01

Distribusi Skor Review:
score
1    14349
2     6883
3     8820
4    17977
5    56182
Name: count, dtype: int64
DataFrame telah disimpan ke nike_reviews.csv


In [15]:
print(f"Jumlah data yang telah di-scraping: {df_nike_reviews.shape[0]}")

Jumlah data yang telah di-scraping: 104211


In [16]:
print("Informasi DataFrame:")
df_nike_reviews.info()

Informasi DataFrame:
<class 'pandas.core.frame.DataFrame'>
Index: 104211 entries, 0 to 127037
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   reviewId              104211 non-null  object        
 1   userName              104211 non-null  object        
 2   userImage             104211 non-null  object        
 3   content               104211 non-null  object        
 4   score                 104211 non-null  int64         
 5   thumbsUpCount         104211 non-null  int64         
 6   reviewCreatedVersion  96609 non-null   object        
 7   at                    104211 non-null  datetime64[ns]
 8   replyContent          32 non-null      object        
 9   repliedAt             32 non-null      datetime64[ns]
 10  appVersion            96609 non-null   object        
dtypes: datetime64[ns](2), int64(2), object(7)
memory usage: 9.5+ MB
